# Phase 3 — Reproduce Sentence-BERT on a T4

Trains the siamese encoder from `src/training/`. The loop itself is **not** in this
notebook: it lives in the repository so it is diffable in git and so the local debug
run and this run execute identical code.

**Runtime → Change runtime type → T4 GPU** before running anything.

Checkpoints go to Google Drive every 1000 steps. If Colab disconnects, re-run the
training cell — it resumes from the last checkpoint rather than starting over.


## 1. Confirm we actually have a GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'


## 2. Mount Drive

Checkpoints must survive the session, not live in Colab's temporary disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
CKPT = pathlib.Path('/content/drive/MyDrive/echo/phase3')
CKPT.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', CKPT)


## 3. Get the code


In [ ]:
%cd /content
![ -d Echo ] && (cd Echo && git pull -q) || git clone -q https://github.com/ayn-aval/Echo.git
%cd /content/Echo
!git log --oneline -1


## 4. Install

Colab already has torch. Only the rest is needed.


In [ ]:
!pip install -q transformers datasets scipy 2>&1 | tail -2
import transformers, datasets
print('transformers', transformers.__version__, '| datasets', datasets.__version__)


## 5. Train

`distilroberta-base`, 300k stratified NLI pairs, **the paper's exact configuration**:
1 epoch, batch 16, Adam at 2e-5, linear warmup over the first 10% of steps, mean
pooling, `(u, v, |u-v|)`.

Batch 16 is small for a T4 and a larger one would be faster — but keeping the paper's
config means that when our numbers land below theirs, batch size and learning rate are
ruled out by construction and the training-subset size is the only difference left to
explain.

**Safe to re-run after a disconnect** — `--resume` picks up from the last checkpoint.


In [ ]:
!python -m src.training.train \
    --model distilroberta-base \
    --pairs 300000 \
    --batch-size 16 \
    --lr 2e-5 \
    --pooling mean \
    --variant 'u,v,|u-v|' \
    --checkpoint-every 1000 \
    --out /content/drive/MyDrive/echo/phase3/sbert-distilroberta \
    --resume


## 6. Evaluate on all seven STS datasets

Uses `evaluate_sts()` from Phase 2 unchanged — the same function that measured the
GloVe and raw-BERT baselines, so the comparison is real rather than merely similar.


In [ ]:
import logging; logging.getLogger('transformers').setLevel(logging.ERROR)
import pandas as pd
from eval.sts_eval import evaluate_sts
from src.embeddings import sbert

ENCODER = '/content/drive/MyDrive/echo/phase3/sbert-distilroberta/encoder'
df = evaluate_sts(sbert.make_encoder(ENCODER), 'sbert-distilroberta-300k')
df.to_csv('/content/drive/MyDrive/echo/phase3/sts_results.csv', index=False)
df


## 7. Side-by-side against the paper's Table 1


In [ ]:
PAPER = {
    'Avg. GloVe (paper)':  [55.14, 70.66, 59.73, 68.25, 63.66, 58.02, 53.76, 61.32],
    'Avg. BERT (paper)':   [38.78, 57.98, 57.98, 63.15, 61.06, 46.35, 58.40, 54.81],
    'BERT CLS (paper)':    [20.16, 30.01, 20.09, 36.88, 38.08, 16.50, 42.63, 29.19],
    'SBERT-NLI-base (paper)': [70.97, 76.53, 73.19, 79.09, 74.30, 77.03, 72.91, 74.89],
    'SRoBERTa-NLI-base (paper)': [71.54, 72.49, 70.80, 78.74, 73.69, 77.77, 74.46, 74.21],
}
COLS = ['STS12','STS13','STS14','STS15','STS16','STS-B','SICK-R','Avg']
table = pd.DataFrame(PAPER, index=COLS).T
mine = df.set_index('dataset').spearman.reindex(COLS)
table.loc['THIS RUN (300k subset)'] = mine.values
gap = table.loc['SBERT-NLI-base (paper)','Avg'] - table.loc['THIS RUN (300k subset)','Avg']
print(f'gap to SBERT-NLI-base: {gap:+.2f} Spearman points')
table.round(2)


## 8. Copy results back into the repo

Download `sts_results.csv` from Drive and commit it, so every number in the README
traces to a file rather than a screenshot.


In [ ]:
!cp /content/drive/MyDrive/echo/phase3/sts_results.csv /content/Echo/results/ 2>/dev/null
!ls -la /content/drive/MyDrive/echo/phase3/
print('\nDownload sts_results.csv and the encoder/ folder from Drive.')
